In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score,
                             roc_curve, classification_report)
from sklearn.inspection import permutation_importance
from scipy import stats

In [2]:
OUTPUT = "D:\Data Analytics\Project 6\Output"
os.makedirs(OUTPUT, exist_ok=True)

In [3]:
# ─── Colour palette ───────────────────────────────────────────────────────────
PALETTE   = ["#1B4F72", "#E74C3C", "#2ECC71", "#F39C12", "#8E44AD",
             "#16A085", "#D35400", "#2C3E50", "#C0392B", "#27AE60"]
FAULT_PAL = {"Fault": "#E74C3C", "Normal": "#2ECC71"}
BLUE      = "#1B4F72"
RED       = "#E74C3C"
GREEN     = "#2ECC71"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

In [4]:
df = pd.read_csv(r"D:\Data Analytics\Project 6\antenna_fault.csv")

print(f"\nShape          : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage   : {df.memory_usage(deep=True).sum()/1024:.1f} KB\n")

feature_desc = {
    "Length"        : "Physical length of the antenna patch (mm)",
    "Width"         : "Physical width of the antenna patch (mm)",
    "Height"        : "Substrate height / thickness (mm)",
    "Permittivity"  : "Relative permittivity of the substrate material",
    "Conductivity"  : "Electrical conductivity of the antenna material (S/m)",
    "Bend"          : "Bend angle / degree of mechanical deformation",
    "Feed"          : "Feed-point impedance or feed offset (Ω or mm)",
    "S11"           : "Return loss — lower (more negative) = better match (dB)",
    "VSWR"          : "Voltage Standing Wave Ratio — ideal = 1 (lower better)",
    "Gain"          : "Antenna directional gain (dBi)",
    "Efficiency"    : "Radiation efficiency (%)",
    "Bandwidth"     : "Operating bandwidth (MHz)",
    "WiFi Fault"    : "WiFi fault label (multi-class)",
    "BT Fault"      : "Bluetooth fault label (multi-class)",
    "WiFi Status"   : "Binary WiFi status: Fault / Normal",
    "BT Status"     : "Binary BT status: Fault / Normal",
    "epsilon_r"     : "Relative permittivity (εr) — substrate dielectric constant",
}
print("Feature Descriptions:")
for col, desc in feature_desc.items():
    print(f"  {col:<18} : {desc}")

print("\nData types:\n", df.dtypes.to_string())

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()
print(f"\nNumerical ({len(num_cols)})  : {num_cols}")
print(f"Categorical ({len(cat_cols)}) : {cat_cols}")

# Class balance
print("\n--- WiFi Fault distribution ---")
print(df["WiFi Fault"].value_counts())
print("\n--- BT Fault distribution ---")
print(df["BT Fault"].value_counts())
print("\n--- WiFi Status distribution ---")
print(df["WiFi Status"].value_counts())
print("\n--- BT Status distribution ---")
print(df["BT Status"].value_counts())


Shape          : 2,688 rows × 17 columns
Memory usage   : 961.5 KB

Feature Descriptions:
  Length             : Physical length of the antenna patch (mm)
  Width              : Physical width of the antenna patch (mm)
  Height             : Substrate height / thickness (mm)
  Permittivity       : Relative permittivity of the substrate material
  Conductivity       : Electrical conductivity of the antenna material (S/m)
  Bend               : Bend angle / degree of mechanical deformation
  Feed               : Feed-point impedance or feed offset (Ω or mm)
  S11                : Return loss — lower (more negative) = better match (dB)
  VSWR               : Voltage Standing Wave Ratio — ideal = 1 (lower better)
  Gain               : Antenna directional gain (dBi)
  Efficiency         : Radiation efficiency (%)
  Bandwidth          : Operating bandwidth (MHz)
  WiFi Fault         : WiFi fault label (multi-class)
  BT Fault           : Bluetooth fault label (multi-class)
  WiFi Status   

In [5]:
# Missing values
missing = df.isnull().sum()
print(f"\nMissing values:\n{missing[missing>0] if missing.any() else '  None found ✓'}")


Missing values:
  None found ✓


In [6]:
# Duplicates
n_dup = df.duplicated().sum()
print(f"Duplicate rows : {n_dup}")
df = df.drop_duplicates()

Duplicate rows : 1636


In [7]:
# Standardise category strings
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

In [10]:
# Binary targets from Status columns
df["WiFi_Binary"] = (df["WiFi Status"] == "Fault").astype(int)
df["BT_Binary"]   = (df["BT Status"]   == "Fault").astype(int)
print("\nBinary target balance:")
print(f"  WiFi Fault rate : {df['WiFi_Binary'].mean()*100:.1f}%")
print(f"  BT   Fault rate : {df['BT_Binary'].mean()*100:.1f}%")


Binary target balance:
  WiFi Fault rate : 97.2%
  BT   Fault rate : 84.9%


In [11]:
# Multi-class integer encoding (for reference)
le_wifi = LabelEncoder()
le_bt   = LabelEncoder()
df["WiFi_Class"] = le_wifi.fit_transform(df["WiFi Fault"])
df["BT_Class"]   = le_bt.fit_transform(df["BT Fault"])

print(f"\nCleaned dataset shape: {df.shape}")


Cleaned dataset shape: (1052, 21)


In [12]:
rf_features = ["Length","Width","Height","Permittivity","Conductivity",
               "Bend","Feed","S11","VSWR","Gain","Efficiency","Bandwidth","epsilon_r"]

print("\nSummary Statistics:")
print(df[rf_features].describe().round(3).to_string())


Summary Statistics:
         Length     Width    Height  Permittivity  Conductivity      Bend      Feed       S11      VSWR      Gain  Efficiency  Bandwidth  epsilon_r
count  1052.000  1052.000  1052.000      1052.000      1052.000  1052.000  1052.000  1052.000  1052.000  1052.000    1052.000   1052.000   1052.000
mean     45.064    34.822     1.196         1.985      8970.671     0.673   -15.136   -13.524     2.613     2.359      59.393     97.428      2.749
std       2.872     2.916     0.228         0.288      3519.623     0.317     5.754     6.504     0.904     2.097      18.063     36.146      0.736
min      40.010    30.010     0.800         1.500      3013.710     0.100   -25.000   -25.000     1.000    -0.990      30.070     40.040      1.500
25%      42.648    32.300     1.008         1.740      5844.448     0.418   -20.125   -19.152     1.810     0.470      42.838     64.240      2.130
50%      45.095    34.680     1.200         1.980      9071.645     0.700   -15.175   -13.2

In [13]:
# ── FIG 1 : Distribution histograms ─────────────────────────────────────────
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
fig.suptitle("Feature Distributions — Antenna Physical & RF Parameters",
             fontsize=16, fontweight="bold", y=1.01)
axes = axes.flatten()
for i, col in enumerate(rf_features):
    ax = axes[i]
    ax.hist(df[col], bins=40, color=PALETTE[i % len(PALETTE)],
            edgecolor="white", linewidth=0.4, alpha=0.85)
    ax.set_title(col, fontweight="bold")
    ax.set_ylabel("Count")
    mu, sd = df[col].mean(), df[col].std()
    ax.axvline(mu, color="black", lw=1.5, ls="--", label=f"μ={mu:.2f}")
    ax.legend(fontsize=8)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig1_distributions.png", bbox_inches="tight")
plt.close()
print("  Saved fig1_distributions.png")

  Saved fig1_distributions.png


In [14]:
# ── FIG 2 : Boxplots by WiFi fault status ────────────────────────────────────
key_rf = ["S11","VSWR","Gain","Efficiency","Bandwidth","Bend","Conductivity","Feed"]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("RF / Physical Parameters vs WiFi Fault Status",
             fontsize=15, fontweight="bold")
axes = axes.flatten()
for i, col in enumerate(key_rf):
    ax = axes[i]
    data_fault  = df[df["WiFi_Binary"]==1][col].dropna()
    data_normal = df[df["WiFi_Binary"]==0][col].dropna()
    bp = ax.boxplot([data_fault, data_normal],
                    patch_artist=True,
                    labels=["Fault","Normal"],
                    medianprops=dict(color="black", linewidth=2))
    bp["boxes"][0].set_facecolor(RED)
    bp["boxes"][0].set_alpha(0.75)
    bp["boxes"][1].set_facecolor(GREEN)
    bp["boxes"][1].set_alpha(0.75)
    ax.set_title(col, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig2_boxplots_wifi.png", bbox_inches="tight")
plt.close()
print("  Saved fig2_boxplots_wifi.png")

  Saved fig2_boxplots_wifi.png


In [15]:
# ── FIG 3 : Boxplots by BT fault status ──────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("RF / Physical Parameters vs BT Fault Status",
             fontsize=15, fontweight="bold")
axes = axes.flatten()
for i, col in enumerate(key_rf):
    ax = axes[i]
    data_fault  = df[df["BT_Binary"]==1][col].dropna()
    data_normal = df[df["BT_Binary"]==0][col].dropna()
    bp = ax.boxplot([data_fault, data_normal],
                    patch_artist=True,
                    labels=["Fault","Normal"],
                    medianprops=dict(color="black", linewidth=2))
    bp["boxes"][0].set_facecolor(RED)
    bp["boxes"][0].set_alpha(0.75)
    bp["boxes"][1].set_facecolor(GREEN)
    bp["boxes"][1].set_alpha(0.75)
    ax.set_title(col, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig3_boxplots_bt.png", bbox_inches="tight")
plt.close()
print("  Saved fig3_boxplots_bt.png")

  Saved fig3_boxplots_bt.png


In [16]:
# ── FIG 4 : Correlation heatmap ───────────────────────────────────────────────
corr_cols = rf_features + ["WiFi_Binary","BT_Binary"]
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = LinearSegmentedColormap.from_list("rf", ["#E74C3C","#FDFEFE","#1B4F72"])
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, mask=mask, cmap=cmap, annot=True, fmt=".2f",
            linewidths=0.5, linecolor="white", ax=ax,
            vmin=-1, vmax=1, square=True, annot_kws={"size":8})
ax.set_title("Pearson Correlation Matrix — RF Features + Fault Targets",
             fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig4_correlation_heatmap.png", bbox_inches="tight")
plt.close()
print("  Saved fig4_correlation_heatmap.png")

  Saved fig4_correlation_heatmap.png


In [17]:
# ── FIG 5 : Fault type breakdown ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Fault Type Distribution — WiFi vs Bluetooth",
             fontsize=15, fontweight="bold")

wifi_counts = df["WiFi Fault"].value_counts()
bars1 = ax1.barh(wifi_counts.index, wifi_counts.values,
                 color=PALETTE[:len(wifi_counts)], edgecolor="white")
ax1.set_title("WiFi Fault Types", fontweight="bold")
ax1.set_xlabel("Count")
for bar in bars1:
    ax1.text(bar.get_width()+5, bar.get_y()+bar.get_height()/2,
             f"{bar.get_width():,.0f}", va="center", fontsize=9)

bt_counts = df["BT Fault"].value_counts()
bars2 = ax2.barh(bt_counts.index, bt_counts.values,
                 color=PALETTE[:len(bt_counts)], edgecolor="white")
ax2.set_title("BT Fault Types", fontweight="bold")
ax2.set_xlabel("Count")
for bar in bars2:
    ax2.text(bar.get_width()+5, bar.get_y()+bar.get_height()/2,
             f"{bar.get_width():,.0f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig5_fault_breakdown.png", bbox_inches="tight")
plt.close()
print("  Saved fig5_fault_breakdown.png")

  Saved fig5_fault_breakdown.png


In [18]:
# ── FIG 6 : S11 vs VSWR coloured by WiFi fault ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Key RF Metrics: S11 vs VSWR", fontsize=14, fontweight="bold")

for ax, (target, label) in zip(axes, [("WiFi_Binary","WiFi"), ("BT_Binary","BT")]):
    for val, color, lbl in [(0, GREEN, "Normal"), (1, RED, "Fault")]:
        sub = df[df[target]==val]
        ax.scatter(sub["S11"], sub["VSWR"], c=color, label=lbl,
                   alpha=0.45, s=15, edgecolors="none")
    ax.set_xlabel("S11 Return Loss (dB)")
    ax.set_ylabel("VSWR")
    ax.set_title(f"{label} Fault — S11 vs VSWR")
    ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig6_s11_vswr_scatter.png", bbox_inches="tight")
plt.close()
print("  Saved fig6_s11_vswr_scatter.png")

  Saved fig6_s11_vswr_scatter.png


In [19]:
# ── FIG 7 : Outlier detection (Z-score) ──────────────────────────────────────
z_scores = np.abs(stats.zscore(df[rf_features]))
outlier_count = (z_scores > 3).sum(axis=0)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(rf_features, outlier_count, color=PALETTE[:len(rf_features)],
              edgecolor="white")
ax.set_title("Outlier Count per Feature (|Z-score| > 3)", fontweight="bold")
ax.set_ylabel("Number of Outliers")
ax.set_xlabel("Feature")
plt.xticks(rotation=30, ha="right")
for bar, val in zip(bars, outlier_count):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            str(val), ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig7_outliers.png", bbox_inches="tight")
plt.close()
print("  Saved fig7_outliers.png")

  Saved fig7_outliers.png


In [20]:
# Efficiency-to-Bandwidth ratio
df["Eff_BW_Ratio"]    = df["Efficiency"] / (df["Bandwidth"] + 1e-9)

In [21]:
# Gain-to-VSWR ratio (higher = better)
df["Gain_VSWR_Ratio"] = df["Gain"] / (df["VSWR"] + 1e-9)

In [22]:
# Return loss magnitude (absolute S11)
df["S11_abs"]         = df["S11"].abs()

In [23]:
# Impedance mismatch proxy: VSWR-1 / VSWR+1 → reflection coefficient
df["Gamma"]           = (df["VSWR"] - 1) / (df["VSWR"] + 1)

In [24]:
# Antenna volume (physical size)
df["Volume"]          = df["Length"] * df["Width"] * df["Height"]

In [25]:
# Normalised feed offset
df["Feed_norm"]       = df["Feed"] / df["Feed"].abs().max()

In [26]:
engineered = ["Eff_BW_Ratio","Gain_VSWR_Ratio","S11_abs","Gamma","Volume","Feed_norm"]
print("  Engineered features:")
for f in engineered:
    print(f"    {f:<20} : {df[f].describe()[['mean','std','min','max']].to_dict()}")

# Final feature set
features = rf_features + engineered

  Engineered features:
    Eff_BW_Ratio         : {'mean': 0.7232260513378426, 'std': 0.40568616306138394, 'min': 0.19044316963884234, 'max': 2.1314313581533426}
    Gain_VSWR_Ratio      : {'mean': 1.0420312626091932, 'std': 1.1189273016695076, 'min': -0.8715596322279268, 'max': 5.37499999483173}
    S11_abs              : {'mean': 13.524220532319392, 'std': 6.503848415078143, 'min': 3.02, 'max': 25.0}
    Gamma                : {'mean': 0.4053280956303249, 'std': 0.16849684404353762, 'min': 0.0, 'max': 0.6}
    Volume               : {'mean': 1878.6708702110266, 'std': 420.5773845884837, 'min': 1103.8376799999999, 'max': 3071.731062}
    Feed_norm            : {'mean': -0.6054330798479087, 'std': 0.23014287302997305, 'min': -1.0, 'max': -0.2}


In [27]:
# 5. STATISTICAL ANALYSIS

In [28]:
print("\n  Pearson correlation with WiFi Fault (binary):")
wifi_corr = df[features + ["WiFi_Binary"]].corr()["WiFi_Binary"].drop("WiFi_Binary").sort_values(key=abs, ascending=False)
print(wifi_corr.round(4).to_string())


  Pearson correlation with WiFi Fault (binary):
Gain_VSWR_Ratio   -0.2358
Gamma              0.1986
VSWR               0.1975
Bandwidth         -0.1548
Efficiency        -0.1442
Gain              -0.1273
S11_abs           -0.1168
S11                0.1168
Volume            -0.0963
Height            -0.0837
Eff_BW_Ratio       0.0577
Width             -0.0503
Conductivity       0.0411
Permittivity       0.0238
Length            -0.0196
Feed_norm         -0.0073
Feed              -0.0073
Bend              -0.0050
epsilon_r         -0.0026


In [29]:
print("\n  Pearson correlation with BT Fault (binary):")
bt_corr = df[features + ["BT_Binary"]].corr()["BT_Binary"].drop("BT_Binary").sort_values(key=abs, ascending=False)
print(bt_corr.round(4).to_string())


  Pearson correlation with BT Fault (binary):
VSWR               0.3458
Gamma              0.3203
Gain_VSWR_Ratio   -0.3028
Bandwidth         -0.1684
S11_abs           -0.1637
S11                0.1637
Efficiency        -0.1589
Gain              -0.1395
Eff_BW_Ratio       0.0847
Volume            -0.0532
Height            -0.0428
Width             -0.0345
Conductivity      -0.0305
Length            -0.0277
Bend              -0.0250
Permittivity      -0.0238
epsilon_r          0.0109
Feed_norm         -0.0072
Feed              -0.0072


In [30]:
# Mann-Whitney U tests (non-parametric) for key features
print("\n  Mann-Whitney U tests (Fault vs Normal) — WiFi:")
mw_results = {}
for col in rf_features:
    group_fault  = df[df["WiFi_Binary"]==1][col].dropna()
    group_normal = df[df["WiFi_Binary"]==0][col].dropna()
    stat, p = stats.mannwhitneyu(group_fault, group_normal, alternative="two-sided")
    mw_results[col] = p
    sig = "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else ""))
    print(f"    {col:<18} p={p:.4e}  {sig}")


  Mann-Whitney U tests (Fault vs Normal) — WiFi:
    Length             p=5.2525e-01  
    Width              p=1.0309e-01  
    Height             p=6.1207e-03  **
    Permittivity       p=4.3865e-01  
    Conductivity       p=1.8025e-01  
    Bend               p=8.3914e-01  
    Feed               p=8.1982e-01  
    S11                p=1.4898e-04  ***
    VSWR               p=7.2137e-10  ***
    Gain               p=4.1782e-05  ***
    Efficiency         p=4.5770e-06  ***
    Bandwidth          p=5.5487e-07  ***
    epsilon_r          p=9.5009e-01  


In [31]:
# WiFi vs BT Fault co-occurrence
co_occur = (df["WiFi_Binary"] & df["BT_Binary"]).sum()
total    = len(df)
chi2, p_chi, *_ = stats.chi2_contingency(
    pd.crosstab(df["WiFi_Binary"], df["BT_Binary"]))
print(f"\n  WiFi & BT fault co-occurrence : {co_occur}/{total} ({co_occur/total*100:.1f}%)")
print(f"  Chi² independence test       : χ²={chi2:.2f}, p={p_chi:.4e}")


  WiFi & BT fault co-occurrence : 893/1052 (84.9%)
  Chi² independence test       : χ²=160.76, p=7.7331e-37


In [32]:
# 6. MACHINE LEARNING

In [33]:
X = df[features].fillna(df[features].median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

models = {
    "Logistic Regression" : LogisticRegression(max_iter=2000, random_state=42, C=1.0),
    "Random Forest"        : RandomForestClassifier(n_estimators=200, random_state=42,
                                                    class_weight="balanced"),
    "Gradient Boosting"    : GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                                         max_depth=4, random_state=42),
}

all_results = {}
all_cm      = {}
all_roc     = {}

for target_name, target_col in [("WiFi Fault", "WiFi_Binary"), ("BT Fault", "BT_Binary")]:
    y = df[target_col].values
    X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2,
                                               random_state=42, stratify=y)
    print(f"\n  {'─'*60}")
    print(f"  Target: {target_name}")
    print(f"  Train size: {len(X_tr):,}  |  Test size: {len(X_te):,}")
    print(f"  {'─'*60}")

    target_results = {}
    target_cm      = {}
    target_roc     = {}

    for model_name, model in models.items():
        model.fit(X_tr, y_tr)
        y_pred  = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:,1]

        acc  = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, zero_division=0)
        rec  = recall_score(y_te, y_pred, zero_division=0)
        f1   = f1_score(y_te, y_pred, zero_division=0)
        auc  = roc_auc_score(y_te, y_proba)
        cm   = confusion_matrix(y_te, y_pred)
        fpr, tpr, _ = roc_curve(y_te, y_proba)

        target_results[model_name] = dict(Accuracy=acc, Precision=prec,
                                          Recall=rec, F1=f1, AUC=auc)
        target_cm[model_name]      = cm
        target_roc[model_name]     = (fpr, tpr, auc)

        print(f"\n  [{model_name}]")
        print(f"    Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f} | AUC={auc:.4f}")

    all_results[target_name] = target_results
    all_cm[target_name]      = target_cm
    all_roc[target_name]     = target_roc


  ────────────────────────────────────────────────────────────
  Target: WiFi Fault
  Train size: 841  |  Test size: 211
  ────────────────────────────────────────────────────────────

  [Logistic Regression]
    Acc=0.9716 | Prec=0.9761 | Rec=0.9951 | F1=0.9855 | AUC=0.9683

  [Random Forest]
    Acc=0.9810 | Prec=0.9809 | Rec=1.0000 | F1=0.9903 | AUC=0.9959

  [Gradient Boosting]
    Acc=0.9810 | Prec=0.9809 | Rec=1.0000 | F1=0.9903 | AUC=0.9764

  ────────────────────────────────────────────────────────────
  Target: BT Fault
  Train size: 841  |  Test size: 211
  ────────────────────────────────────────────────────────────

  [Logistic Regression]
    Acc=0.9005 | Prec=0.9115 | Rec=0.9777 | F1=0.9434 | AUC=0.9188

  [Random Forest]
    Acc=1.0000 | Prec=1.0000 | Rec=1.0000 | F1=1.0000 | AUC=1.0000

  [Gradient Boosting]
    Acc=1.0000 | Prec=1.0000 | Rec=1.0000 | F1=1.0000 | AUC=1.0000


In [34]:
# ── FIG 8 : Confusion matrices ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Confusion Matrices — WiFi (top) & BT (bottom) Fault Prediction",
             fontsize=14, fontweight="bold")
cmap_cm = LinearSegmentedColormap.from_list("cm", ["#FDFEFE","#1B4F72"])
for row, target_name in enumerate(["WiFi Fault","BT Fault"]):
    for col_i, model_name in enumerate(models.keys()):
        ax = axes[row][col_i]
        cm = all_cm[target_name][model_name]
        sns.heatmap(cm, annot=True, fmt="d", cmap=cmap_cm, ax=ax,
                    linewidths=1, linecolor="white",
                    xticklabels=["Normal","Fault"],
                    yticklabels=["Normal","Fault"],
                    cbar=False, annot_kws={"size":13})
        ax.set_title(f"{target_name}\n{model_name}", fontweight="bold", fontsize=10)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig8_confusion_matrices.png", bbox_inches="tight")
plt.close()
print("\n  Saved fig8_confusion_matrices.png")


  Saved fig8_confusion_matrices.png


In [35]:
# ── FIG 9 : ROC curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("ROC Curves — All Models", fontsize=14, fontweight="bold")
for ax, target_name in zip(axes, ["WiFi Fault","BT Fault"]):
    for (model_name, (fpr, tpr, auc)), color in zip(
            all_roc[target_name].items(), PALETTE):
        ax.plot(fpr, tpr, lw=2.5, color=color,
                label=f"{model_name} (AUC={auc:.3f})")
    ax.plot([0,1],[0,1],"--", color="gray", lw=1.5, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{target_name}", fontweight="bold")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig9_roc_curves.png", bbox_inches="tight")
plt.close()
print("  Saved fig9_roc_curves.png")

  Saved fig9_roc_curves.png


In [36]:
# ── FIG 10 : Model comparison bar chart ──────────────────────────────────────
metrics_to_plot = ["Accuracy","Precision","Recall","F1","AUC"]
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Model Performance Comparison", fontsize=14, fontweight="bold")
for ax, target_name in zip(axes, ["WiFi Fault","BT Fault"]):
    res = all_results[target_name]
    model_names = list(res.keys())
    x = np.arange(len(model_names))
    w = 0.15
    for i, metric in enumerate(metrics_to_plot):
        vals = [res[m][metric] for m in model_names]
        bars = ax.bar(x + i*w - 2*w, vals, w, label=metric,
                      color=PALETTE[i], edgecolor="white", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=10, ha="right")
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Score")
    ax.set_title(target_name, fontweight="bold")
    ax.legend(loc="upper left", fontsize=8)
    ax.axhline(0.8, ls="--", color="gray", lw=1, alpha=0.6)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig10_model_comparison.png", bbox_inches="tight")
plt.close()
print("  Saved fig10_model_comparison.png")

  Saved fig10_model_comparison.png


In [37]:
# 7. FEATURE IMPORTANCE

In [38]:
fi_results = {}
for target_name, target_col in [("WiFi Fault","WiFi_Binary"), ("BT Fault","BT_Binary")]:
    y  = df[target_col].values
    X_ = df[features].fillna(df[features].median())
    X_sc = scaler.transform(X_)
    rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
    rf.fit(X_sc, y)
    fi = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
    fi_results[target_name] = fi
    print(f"\n  Top-10 features for {target_name}:")
    print(fi.head(10).round(4).to_string())


  Top-10 features for WiFi Fault:
VSWR               0.1813
Gamma              0.1731
Efficiency         0.1413
Gain_VSWR_Ratio    0.1411
Bandwidth          0.1087
Gain               0.0732
S11_abs            0.0519
S11                0.0403
Eff_BW_Ratio       0.0397
Volume             0.0130

  Top-10 features for BT Fault:
Gamma              0.1695
VSWR               0.1683
Efficiency         0.1351
Bandwidth          0.1331
Gain_VSWR_Ratio    0.0868
Gain               0.0754
S11                0.0735
S11_abs            0.0694
Eff_BW_Ratio       0.0414
Volume             0.0067


In [39]:
# ── FIG 11 : Feature importance ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle("Random Forest Feature Importance", fontsize=14, fontweight="bold")
for ax, target_name in zip(axes, ["WiFi Fault","BT Fault"]):
    fi = fi_results[target_name].head(15)
    colors_ = [PALETTE[i % len(PALETTE)] for i in range(len(fi))]
    bars = ax.barh(fi.index[::-1], fi.values[::-1], color=colors_[::-1],
                   edgecolor="white")
    ax.set_title(f"{target_name}", fontweight="bold")
    ax.set_xlabel("Importance Score")
    for bar in bars:
        ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
                f"{bar.get_width():.4f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig11_feature_importance.png", bbox_inches="tight")
plt.close()
print("\n  Saved fig11_feature_importance.png")


  Saved fig11_feature_importance.png


In [40]:
# ── FIG 12 : Top features by fault type (violin) ─────────────────────────────
top5 = fi_results["WiFi Fault"].head(5).index.tolist()
fig, axes = plt.subplots(1, 5, figsize=(22, 6))
fig.suptitle("Top-5 Features — Distribution by WiFi Fault Status",
             fontsize=13, fontweight="bold")
for ax, col in zip(axes, top5):
    parts = ax.violinplot([df[df["WiFi_Binary"]==0][col].dropna(),
                           df[df["WiFi_Binary"]==1][col].dropna()],
                          showmedians=True)
    for pc, col_ in zip(parts["bodies"], [GREEN, RED]):
        pc.set_facecolor(col_)
        pc.set_alpha(0.7)
    ax.set_xticks([1,2])
    ax.set_xticklabels(["Normal","Fault"])
    ax.set_title(col, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig12_violin_top_features.png", bbox_inches="tight")
plt.close()
print("  Saved fig12_violin_top_features.png")

  Saved fig12_violin_top_features.png
